# 02 - r1 theta sweep (remaining cells) + merge

Runs only the discipline x theta cells without a partial JSON in the bundle
(local state: econ and math complete, physics 4/5, neuro 3/5, chemistry 2/5).
Edit `FIELDS` below to split disciplines across two runtimes if you want.
CPU-bound boosting; the A100 runtime's CPUs are fine.

In [ ]:
import sys
sys.path.insert(0, "/content/drive/MyDrive/who-inherits")  # for colab_common if bundle not yet unzipped
try:
    import colab_common as cc
except ImportError:
    # colab_common ships inside the bundle; bootstrap: mount, unzip, import
    from google.colab import drive as _d; _d.mount("/content/drive")
    import subprocess
    subprocess.run(["unzip", "-q", "-o",
                    "/content/drive/MyDrive/who-inherits/colab_bundle.zip",
                    "-d", "/content/work"], check=True)
    sys.path.insert(0, "/content/work/colab")
    import colab_common as cc
else:
    cc.mount_drive()
sys.path.insert(0, "/content/work/colab")
import colab_common as cc
cc.setup_workspace()
cc.verify_frozen_hashes()
print(cc.run_meta())

In [ ]:
def field_env(field):
    return {"DATASET": field,
            "DATASET_PATH": f"data/clean_dataset_{field}.parquet"}

FIELDS = ["physics", "neuro", "chemistry"]   # econ + math already complete

In [ ]:
ckpt = cc.start_checkpoint_thread()
for f in FIELDS:
    cc.run_script("code/paper_pipeline/experiments/r1_theta_sweep.py",
                  env_extra=field_env(f))   # skips completed theta cells
    cc.sync_to_drive()                       # per-discipline unit sync
cc.stop_checkpoint_thread()

In [ ]:
# merge: all 25 cells (bundled + new) -> theta_<t>.json + summary
from pathlib import Path
parts = sorted(p.name for p in Path("/content/work/results/robustness/theta_partial").glob("*.json"))
assert len(parts) == 25, f"expected 25 theta cells, have {len(parts)}: {parts}"
cc.run_script("code/paper_pipeline/experiments/r1_theta_sweep.py", args=("--merge",))
import json
s = json.load(open("/content/work/results/robustness/theta_sweep_summary.json"))
for f, blk in s["fields"].items():
    print(f, "stable(all):", blk["branch_stable_all_theta"],
          "| stable(adjacent):", blk["branch_stable_adjacent"])

In [ ]:
cc.write_done_flag("theta")